# 게이트 없는 저차원 N/V 표현 M2 — Dunnhumby seed 42

새 M2만 100 epoch 학습합니다. 기존 historical backtest의 `M1@64` 결과는 데이터 manifest·분할·seed·epoch·학습 설정이 모두 같을 때만 재사용합니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 표현: ID 64차원 + 거래활동 4차원 + 거래당 가치 4차원
- 제거: 사용자 게이트, 학습형 축 가중치, 명시적 아이템 인기도·가격 입력

이 실행은 탐색적 seed 42 screen이며 유의성을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '85b23cf10673d072714dffa093a6d0064fdd645f'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_gatefree_lowdim import (
    configure_gatefree_lowdim_run,
    preflight_summary,
    run_gatefree_lowdim_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_gatefree_lowdim_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_gatefree_lowdim_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)